<div style="background:linear-gradient(120deg,#07062e 0%,#120c5c 60%,#2a1a9e 100%);border-radius:14px;padding:26px 30px;color:#ffffff;font-family:'Segoe UI',Arial,sans-serif;"><div style="font-size:12px;letter-spacing:3px;color:#4de1ff;font-weight:600;">MICROSOFT IGNITE 2026 &nbsp;&middot;&nbsp; LAB663</div><div style="font-size:30px;font-weight:700;margin:8px 0 4px 0;color:#ffffff;">Module 6 &middot; Make it optimal</div><div style="font-size:15px;color:#cfd4ff;">The hill climb &middot; about 14 minutes</div><div style="margin-top:16px;"><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">1</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">2</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">3</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">4</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">5</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#4de1ff;color:#07062e;">6</span></div><div style="height:6px;border-radius:3px;margin-top:18px;background:linear-gradient(90deg,#3cf26b,#4de1ff,#6c4dff,#ff4fd8,#ff9f1c,#ffd23f);"></div></div>

# 6. Make it optimal (v2 to v3)

**Time:** about 14 minutes. The last climb.

**The question we're starting with:** can I cut my model bill without losing quality?

We have a measured baseline, whether Module 5 promoted a rewrite or kept v1. **Every question still goes to one frontier model.** Could routing improve its cost or latency without an unacceptable loss of quality?

This time we change the **model**, not the instructions. We'll swap our single model for a **model router**: one endpoint that picks a model for each request. Then we'll ask the same hard question we always ask:

> **Did quality hold, and did the cost go down?** And we'll answer with the same rubric.

We deployed **four router editions**. The guided core compares `mr-default`; the extended lab compares all four. Every measured version still gets the same twelve questions and two repeats.

| Deployment | How it's set up |
|---|---|
| `mr-default` | The router's own defaults |
| `mr-cost` | Leans toward cheaper models |
| `mr-quality` | Leans toward stronger models |
| `mr-subset` | Only picks from a list we chose |

| Our name | What changes |
|---|---|
| v2 checkpoint | The retained, measured baseline from Module 5 |
| v3 | **Today:** same instructions, right-sized model |


---

## 6.1 Where did we leave off?

Module 5 saved a few facts (our rubric, our datasets, our scores) so we don't have to redo them.

In [1]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from the LAB663 repository.")
sys.path.insert(0, str(root / "src"))
import lab

project_client, openai_client = lab.connect()
state = lab.load_state()
if not {"evaluation_id", "v2_version", "quality_baseline_dev", "quality_baseline_test"} <= state.keys():
    raise RuntimeError("Finish notebook 05, including its promotion decision, before continuing.")

dev_cases, test_cases = lab.load_cases("dev"), lab.load_cases("test")
versions = {int(v.version): v for v in project_client.agents.list_versions(lab.AGENT_NAME)}
v2 = versions[int(state["v2_version"])]
print(f"Retained baseline = Foundry version {v2.version}, model {v2.definition.model}")

# Declare the experiment and teaching policy BEFORE looking at any router scores.
RUN_ALL_EDITIONS = False  # True is the extended lab; it adds six evaluation runs.
EDITIONS = lab.ROUTERS if RUN_ALL_EDITIONS else ["mr-default"]
TOLERANCE = 0.05  # Maximum absolute loss overall AND in each difficulty/failure category.
if "router_tolerance" in state and state["router_tolerance"] != TOLERANCE:
    raise RuntimeError("The promotion tolerance is frozen for this comparison. Do not change it after seeing scores.")
if "router_editions" in state and state["router_editions"] != EDITIONS:
    raise RuntimeError("Choose the router editions before starting a comparison, not after seeing its results.")
lab.save_state(router_tolerance=TOLERANCE, router_editions=EDITIONS)
snapshot = lab.record_deployment_configuration()
print("Actual routing configuration:", snapshot)
print("Editions:", EDITIONS, "| quality tolerance:", TOLERANCE)
print("Guard questions must not regress; unknown cost cannot earn a cost-based promotion.")

Retained baseline = Foundry version 2, model gpt-5.4
Actual routing configuration: {'recorded_at': '2026-10-18T17:56:51.173825+00:00', 'deployments': [{'model': {'callRateLimit': None, 'format': 'OpenAI', 'name': 'gpt-5.4', 'publisher': None, 'source': None, 'sourceAccount': None, 'version': '2026-03-05'}, 'name': 'gpt-5.4', 'routing': None, 'sku': {'capacity': 250, 'family': None, 'name': 'GlobalStandard', 'size': None, 'tier': None}}, {'model': {'callRateLimit': None, 'format': 'OpenAI', 'name': 'gpt-5.4-mini', 'publisher': None, 'source': None, 'sourceAccount': None, 'version': '2026-03-17'}, 'name': 'gpt-5.4-mini', 'routing': None, 'sku': {'capacity': 150, 'family': None, 'name': 'GlobalStandard', 'size': None, 'tier': None}}, {'model': {'callRateLimit': None, 'format': 'OpenAI', 'name': 'model-router', 'publisher': None, 'source': None, 'sourceAccount': None, 'version': '2025-11-18'}, 'name': 'mr-cost', 'routing': {'mode': 'cost', 'models': None}, 'sku': {'capacity': 150, 'family'

## 6.2 Am I overpaying?

A router may help if smaller models can answer part of our workload well. That is something to measure, not assume. Let's count the difficulty tags in our teaching set.

**Predict:** what share of our questions are `easy`?

In [2]:
mix = Counter(case["difficulty"] for case in test_cases)
print(dict(mix))
print(f"Easy questions: {mix['easy']} of {len(test_cases)}")

{'easy': 5, 'medium': 5, 'hard': 2}
Easy questions: 5 of 12


**Think:** only five of our twelve test questions are tagged easy. This deliberately balanced teaching set is not a production traffic estimate. What customer traffic would we need to sample before making a business forecast?

## 6.3 How does a router decide?

Before we trust a router, let's *watch* one decide. We'll send easy and hard questions to each edition and look at which model answered. The reply tells us.

In [3]:
probes = {
    "easy": "How much does the TrailLite Daypack weigh?",
    "hard": (
        "A family of six is camping in heavy rain, wants the lightest tent possible, "
        "and has a tight budget. Weigh the trade-offs and recommend one."
    ),
}

rows = []
for edition in EDITIONS:
    for level, question in probes.items():
        context = lab.manual_text(16) + "\n" + lab.manual_text(1) + "\n" + lab.manual_text(8)
        reply = openai_client.responses.create(model=edition, input=f"{context}\n\nQuestion: {question}")
        # reply.model is the model the router actually used.
        rows.append({"edition": edition, "question": level, "router chose": reply.model})

pd.DataFrame(rows).pivot(index="edition", columns="question", values="router chose")

question,easy,hard
edition,,
mr-default,gpt-5.6-luna-2026-07-09,gpt-5.6-luna-2026-07-09


**Expected:** one row per selected edition, with a serving model for each probe. These direct calls include manuals in the prompt, not the agent's complete file-search workflow; they illustrate routing but do not predict the agent's model distribution.

**Go deeper:** all four `mr-` deployments use a routing mode and, for `mr-subset`, an allowed model list. Defaults are in `infra/main.bicep`; the lifecycle records the deployed configuration. The router's active version can gain underlying models without changing its version string, so save the date and observed model distribution too.

**Think:**
- Did `mr-cost` choose differently from `mr-quality`?
- Did any edition send the easy question to a small model and the hard one to a bigger one?
- Does `mr-subset` ever pick a model outside the list we gave it?

This is the thing our hand-built router from Module 3 was trying to do. Here the platform keeps it up to date for us.

## 6.4 How do I test each router edition fairly?

Now the careful part. We keep the retained baseline's instructions and tools. We change **only the model**, once per selected edition. The controlled change is routing; random variation and service failures still need to be measured.

In [4]:
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition

state = lab.load_state()
saved_candidates = state.get("router_candidates", {})
candidates = {}
for edition in EDITIONS:
    name = f"{lab.AGENT_NAME}-{edition}-candidate"
    previous = saved_candidates.get(edition)
    if previous:
        if previous["baseline_version"] != str(v2.version):
            raise RuntimeError("The router candidate belongs to another baseline. Start a new comparison.")
        saved = next(v for v in project_client.agents.list_versions(name) if str(v.version) == previous["version"])
    else:
        saved = project_client.agents.create_version(name, definition=PromptAgentDefinition(
            model=edition, instructions=v2.definition.instructions, tools=v2.definition.tools))
        saved_candidates[edition] = {"name": name, "version": str(saved.version), "baseline_version": str(v2.version)}
        lab.save_state(router_candidates=saved_candidates)
    expected = PromptAgentDefinition(model=edition, instructions=v2.definition.instructions, tools=v2.definition.tools)
    if saved.definition.as_dict() != expected.as_dict():
        raise RuntimeError("The router candidate changed more than its model. Do not compare this version.")
    candidates[edition] = saved
    print(f"Staged {name}, version {saved.version}; the main agent is unchanged.")

Staged trailmate-mr-default-candidate, version 1; the main agent is unchanged.


## 6.5 Which edition holds quality for less?

We select on **dev**, not `test`, with the same pinned rubric and two complete repeats. Only the selected candidate gets a final held-out check. Repeatedly choosing winners on `test` would turn it into another tuning set.

The board separates quality, errors, agent tokens, estimated model dollars and request latency. Tokens are not dollars; total batch waiting time is not user-facing latency. Missing prices or unaccounted model usage leave cost **unknown**. The p95 is descriptive for this small sample, not a production latency promise.

While it runs, try **Route detective** in [the bonus notebook](../bonus/wait-time-challenges.ipynb). Rerunning this cell resumes the saved run IDs rather than spending on another batch.

In [5]:
runs = {
    edition: lab.run_repeated_eval(
        openai_client, state["evaluation_id"], state["dev_dataset_id"], saved_candidates[edition]["name"],
        candidates[edition].version, f"v3 candidate {edition} on dev", cases=dev_cases,
    )
    for edition in EDITIONS
}

v2_row = state["quality_baseline_dev"]
lab.require_valid(v2_row)
board = pd.DataFrame(
    [{"version": name, **{key: r[key] for key in
       ("valid", "score", "spread", "errored", "tokens", "dollars", "p50_seconds", "p95_seconds")},
      "final models": r["model_counts"]}
     for name, r in [("retained baseline on dev", v2_row), *runs.items()]]
)
display(board.round(3))
lab.save_state(router_dev_scores=runs, router_tolerance=TOLERANCE)

Started v3 candidate mr-default on dev: evalrun_f1ba0e5a502d455da8037c8794bd6a1e
Started v3 candidate mr-default on dev: evalrun_350b11afb1a84c84854452cca2f4ccae
0s waiting; 0/2 finished. evalrun_f1ba0e5a502d455da8037c8794bd6a1e: in_progress; evalrun_350b11afb1a84c84854452cca2f4ccae: in_progress
16s waiting; 0/2 finished. evalrun_f1ba0e5a502d455da8037c8794bd6a1e: in_progress; evalrun_350b11afb1a84c84854452cca2f4ccae: in_progress
32s waiting; 0/2 finished. evalrun_f1ba0e5a502d455da8037c8794bd6a1e: in_progress; evalrun_350b11afb1a84c84854452cca2f4ccae: in_progress
47s waiting; 0/2 finished. evalrun_f1ba0e5a502d455da8037c8794bd6a1e: in_progress; evalrun_350b11afb1a84c84854452cca2f4ccae: in_progress
63s waiting; 0/2 finished. evalrun_f1ba0e5a502d455da8037c8794bd6a1e: in_progress; evalrun_350b11afb1a84c84854452cca2f4ccae: in_progress
79s waiting; 0/2 finished. evalrun_f1ba0e5a502d455da8037c8794bd6a1e: in_progress; evalrun_350b11afb1a84c84854452cca2f4ccae: in_progress
95s waiting; 0/2 finish

,version,valid,score,spread,errored,tokens,dollars,p50_seconds,p95_seconds,final models
0,retained baseline on dev,True,0.807,0.029,0,180878.0,None,5.526,9.168,{'gpt-5.4': 24}
1,mr-default,False,NaN,NaN,6,NaN,None,NaN,NaN,"{'gpt-5.6-luna-2026-07-09': 10, 'gpt-6-sol-202..."


**Expected:** a baseline row and one row per selected edition. `valid=False` means the batch cannot support a comparison or promotion. The helper prints the failed run IDs and saves full output items in `.lab-results/`.

**Read it like a detective.**
- Did every repeat score all twelve questions? Failed requests can make a batch look artificially cheap.
- Is the score gap small compared with the observed repeat spread? Two repeats cannot establish statistical significance.
- Does the `models` column match what you saw in section 3?

Estimated model dollars require verified, region-matching prices for **every** model, including cached input. We subtract measured judge usage and reconcile the model totals before pricing the agent. We never price an entire router call at the final model's rate or treat an unknown price as zero. Tools, storage and the evaluation itself remain separate charges.

## 6.6 Where does the cheap edition slip?

An average hides a lot. Let's break each edition's score down by **difficulty**. A cheap model might ace the easy questions and quietly fail the hard ones.

In [6]:
by_level = pd.DataFrame(
    {name: lab.scores_by(r["per_item"], dev_cases, "difficulty") for name, r in runs.items() if r["valid"]}
)
by_level["v2"] = pd.Series(lab.scores_by(v2_row["per_item"], dev_cases, "difficulty"))
display(by_level.reindex(["easy", "medium", "hard"]).round(3))

evidence = pd.DataFrame([{"edition": name, **item} for name, r in runs.items() for item in r["items"]])
hard_ids = {case["id"] for case in dev_cases if case["difficulty"] == "hard"}
if evidence.empty:
    print("No per-case evidence was returned. Inspect the invalid-batch messages; do not promote.")
else:
    display(evidence[evidence["error"].notna() | evidence["id"].isin(hard_ids)]
            [["edition", "id", "model", "score", "error", "reason"]])

,v2
easy,0.803
medium,0.818
hard,0.798


,edition,id,model,score,error,reason
2,mr-default,dv-03,gpt-5.6-luna-2026-07-09,0.687500,NaN,The verdict is driven mainly by grounded_factu...
6,mr-default,dv-07,,NaN,"EvalAPIError(code='TARGET_INVOCATION_FAILED', ...",NaN
8,mr-default,dv-09,,NaN,"EvalAPIError(code='TARGET_INVOCATION_FAILED', ...",NaN
10,mr-default,dv-11,,NaN,"EvalAPIError(code='TARGET_INVOCATION_FAILED', ...",NaN
11,mr-default,dv-12,gpt-5.6-terra-2026-07-09,0.776786,NaN,The verdict is driven mainly by strong grounde...
14,mr-default,dv-03,gpt-5.6-luna-2026-07-09,0.696970,NaN,The verdict is mainly driven by grounded_factu...
18,mr-default,dv-07,,NaN,"EvalAPIError(code='TARGET_INVOCATION_FAILED', ...",NaN
20,mr-default,dv-09,,NaN,"EvalAPIError(code='TARGET_INVOCATION_FAILED', ...",NaN
22,mr-default,dv-11,,NaN,"EvalAPIError(code='TARGET_INVOCATION_FAILED', ...",NaN
23,mr-default,dv-12,gpt-5.6-terra-2026-07-09,0.803571,NaN,The verdict is driven mainly by strong grounde...


**Think:** is a low score a bad answer, missing retrieval evidence, an unsuitable judge criterion, or a failed request? Inspect the saved tool output and per-dimension reasons before changing routing mode. Difficulty labels alone do not explain the result, and easy questions can fail too.

## 6.7 How do I decide without fooling myself?

Here's a discipline worth keeping: **decide the rule before looking at the winner.** Ours:

> First require complete coverage and the quality rule declared in section 6.1, including each category and guard case. Prefer lower **verified estimated model cost**, then check one selected candidate on held-out questions. Unknown prices cannot establish a cost win.

If none qualify, we keep v2. That's a legitimate result too.

In [7]:
qualified = {}
for name, row in runs.items():
    failures = lab.quality_failures(row, v2_row, dev_cases, TOLERANCE)
    print(name, failures or "Passed the dev quality checks; not yet a promotion.")
    if not failures:
        qualified[name] = row

priced = {name: row for name, row in qualified.items() if row["dollars"] is not None
          and v2_row["dollars"] is not None and row["dollars"] < v2_row["dollars"]}
selected = min(priced, key=lambda name: priced[name]["dollars"]) if priced else next(iter(qualified), None)
if selected:
    print("Candidate for one held-out check:", selected)
    if not priced:
        print("No demonstrated cost saving. This is a quality investigation only, not a cost winner.")
else:
    print("No candidate passed the dev quality checks. We keep the baseline.")

mr-default ['Incomplete evaluation coverage.']
No candidate passed the dev quality checks. We keep the baseline.


### Check the selected candidate once on held-out questions

Do not use this result to try another edition or loosen the threshold. A failed held-out check means we keep the baseline and plan a new experiment. Set the human-review flag only after inspecting the actual answers and judge evidence.

In [8]:
REVIEW_APPROVED = False
winner, selected_test = None, None
v2_test = state["quality_baseline_test"]
if selected:
    previous_selection = lab.load_state().get("router_selected")
    if previous_selection and previous_selection != selected:
        raise RuntimeError("A different candidate has already seen this held-out set. Keep that decision; do not shop for a test winner.")
    lab.save_state(router_selected=selected)
    selected_test = lab.run_repeated_eval(
        openai_client, state["evaluation_id"], state["test_dataset_id"], saved_candidates[selected]["name"],
        candidates[selected].version, f"v3 selected {selected} on test", cases=test_cases)
    failures = lab.quality_failures(selected_test, v2_test, test_cases, TOLERANCE)
    cheaper = (selected_test["dollars"] is not None and v2_test["dollars"] is not None
               and selected_test["dollars"] < v2_test["dollars"])
    selection = {"baseline_version": str(v2.version), "edition": selected,
                 "candidate_version": str(candidates[selected].version)}
    receipt = lab.load_state().get("router_promotion", {})
    approved = REVIEW_APPROVED or receipt.get("selection") == selection
    if not failures and cheaper and selected in priced and approved:
        winner = selected
    print("Held-out decision:", failures or "Quality held within our teaching tolerance.")
    print("Cost saving demonstrated:", cheaper, "| Human review approved:", approved)
    lab.save_state(router_test=selected_test)

**Expected:** an explicit decision. Invalid coverage, a category regression, unknown cost or pending review prevents promotion.

**Think:** would the old rule, based only on average score and total tokens, have hidden a customer-facing failure? Keep the tolerance frozen for this experiment; explore alternative policies only in the local bonus exercise.

## 6.8 How do I make the winner official?

Promotion saves a version only after the checks pass. We record its exact ID and use that ID when invoking it; merely creating a newer candidate is not a release.

In [9]:
if winner:
    saved_state = lab.load_state()
    receipt = saved_state.get("router_promotion", {})
    if receipt.get("selection") == selection:
        v3 = next(v for v in project_client.agents.list_versions(lab.AGENT_NAME) if str(v.version) == receipt["version"])
    else:
        v3 = project_client.agents.create_version(lab.AGENT_NAME, definition=PromptAgentDefinition(
            model=winner, instructions=v2.definition.instructions, tools=v2.definition.tools))
        lab.save_state(router_promotion={"selection": selection, "version": str(v3.version)})
    if v3.definition.as_dict() != candidates[winner].definition.as_dict():
        raise RuntimeError("The promoted router differs from the measured candidate. Stop before using it.")
    v3_version = str(v3.version)
    print(f"v3 = Foundry version {v3_version}, running on {winner}")
    lab.save_state(v3_version=v3_version, v3_edition=winner, v3_for_baseline=str(v2.version), v3_promoted=True)
else:
    lab.save_state(v3_version=str(v2.version), v3_edition=v2.definition.model, v3_promoted=False)
    print("Nothing earned promotion. The retained baseline stays in place.")

Nothing earned promotion. The retained baseline stays in place.


## 6.9 What does the whole climb look like?

The same held-out questions, the same rubric, and six failure categories. Cells may improve, stay flat or get worse. A candidate column is not a claim that we promoted it.

In [10]:
scores = state["test_scores"]
picture = pd.DataFrame({
    "v0": lab.scores_by_way(scores["v0 on test"]["per_item"], test_cases),
    "v1": lab.scores_by_way(scores["v1 on test"]["per_item"], test_cases),
    "v2 checkpoint": lab.scores_by_way(state["quality_baseline_test"]["per_item"], test_cases),
})
checkpoint = selected_test if winner else state["quality_baseline_test"]
picture["v3 checkpoint"] = lab.scores_by_way(checkpoint["per_item"], test_cases)
if selected_test and selected_test["valid"] and not winner:
    picture["router candidate (not promoted)"] = lab.scores_by_way(selected_test["per_item"], test_cases)

display(picture.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format("{:.2f}"))

,v0,v1,v2 checkpoint,v3 checkpoint
wrong_product,0.80,0.78,0.78,0.78
overclaiming,0.81,0.80,0.80,0.80
making_things_up,0.72,0.77,0.77,0.77
wrong_premise,0.76,0.76,0.76,0.76
out_of_lane,0.26,0.46,0.46,0.46
not_useful,0.80,0.77,0.77,0.77


**Think:**
- Which row lit up the most between v0 and v2? That's the **quality** climb: instructions.
- Did v3 hold the line? That's the **cost** climb: the router.
- Which row is still the lowest? That's where our next experiment should go.

## 6.10 Is this the end?

A climb like this isn't one-and-done. Real agents get new questions every day. The same loop keeps going:

1. **Watch** production traffic with insights and continuous evaluation.
2. **Harvest** the questions that went wrong and add them to our test set as new, harder cases.
3. **Version the expanded dataset**, then remeasure every baseline with the same rubric. Scores on different question sets are not directly comparable.
4. **Climb** again, one lever at a time.

**Never shrink the test set or loosen the rubric to bring a score back up.** Fix the agent instead.

## 6.11 What should I try next?

- **What if** we used `mr-cost` everywhere? Where does it break?
- **What if** we used `mr-subset`, a hand-picked list of models? Does it beat the default?
- **What if** we asked the agent optimizer to try **three router options** itself? (It can search over models, not just instructions: set `target_attributes` to include `model` and list the editions in `model_search_space`.)

## 6.12 Where did the savings actually come from? (the aha)

We started by asking whether we can cut the bill without losing quality. One more look, this time at *where* the winner differs from v2.

In [11]:
# Selected candidate minus the retained baseline, using the SAME held-out questions.
if selected_test and selected_test["valid"]:
    after = pd.Series(lab.scores_by(selected_test["per_item"], test_cases, "difficulty"))
    before = pd.Series(lab.scores_by(v2_test["per_item"], test_cases, "difficulty"))
    print((after - before).reindex(["easy", "medium", "hard"]).round(3))
else:
    print("No complete selected-candidate test result; there is no comparable gap to show.")

No complete selected-candidate test result; there is no comparable gap to show.


> **Aha:** Routing is a hypothesis, not a guaranteed saving. Quality, reliability, usage and latency can move in different directions. Complete per-case evidence and verified prices tell us whether the trade-off is acceptable for our workload; an average alone cannot.

We also took away:

- **Our teaching mix is not production traffic.** We need a representative sample before forecasting savings.
- **Reliability is part of the result.** A failed request is not a cheap answer, and a small sample cannot establish production readiness.
- A good promotion rule is **written before looking at the winner**.
- **One lever at a time** kept every result explainable, across all three climbs.
- Insights showed us the slope, the rubric told us the height, the optimizer climbed on quality and the router climbed on cost.

**Go further (optional):** point the optimizer at the model lever. Set `target_attributes` to `["model"]` and list the four router editions in `model_search_space`. Does it pick the same edition we did?

**Nice work. We built it, measured it, broke it, fixed it and understood why.**